# LOOKOUT VO — heading votes from VGGT-1B (Colab / Kaggle, GPU)

Camera poses from **VGGT-1B** (frozen, public weights; code `facebookresearch/vggt@a288dd0`, weights `facebook/VGGT-1B` revision
`860abec`, CC BY-NC 4.0) on the same ≈8 Hz half-resolution frames as the VO tool (the starter's decoder). Keyframe windows
(`kf4_w17`: every 4th step, 17 frames); per window the model returns [R|t] of each frame relative to the first. Output:
`<window>/<clip>.npz` (`ext`, `intr`, `steps`), read by `course_combo.vggt_steps`.

Default configuration = the input of the final entry: the 19 wide and flir-VIS test clips, `kf4_w17`.

**Runtime → Change runtime type → GPU (T4)**, then *Run all*. `SMOKE = True`: one clip, 3 windows. Results go to Google Drive
(`MyDrive/lookout/vggt_runs/<RUN_NAME>`); after a disconnect *Run all* again, finished clips are skipped. The last cells pack
the results and write the environment lock file.

In [ ]:
# ============================ CONFIG ============================
RUN_NAME = 'vggt_v1'
GROUPS = ['test:wide', 'test:flir-VIS']   # order of work; 'split:camera', '*' = every camera
CLIPS = []                       # explicit clip keys override GROUPS
WINDOWS = {'kf4_w17': (4, 17)}   # name: (keyframe spacing in ~8 Hz steps, frames per window)
MASK_WATER = False               # True: pixels below the detected horizon (+6 px) set to 0 before VGGT
BATCH = 2                        # windows per forward pass (T4: 2 is safe)
SMOKE = False                    # True: first flir-VIS train clip, 3 windows per setting, nothing on Drive
SHARD_INDEX, SHARD_COUNT = 0, 1  # several notebooks on one RUN_NAME: same SHARD_COUNT, different SHARD_INDEX
SAVE_TO_DRIVE = True
DATA_URL = 'https://macvi.org/downloads/lookout/v0.1/lookout-vo-dev-v0.1.tar'
VGGT_COMMIT = 'a288dd0f14786c93483e45524328726ab7b1b4ce'
VGGT_WEIGHTS = ('facebook/VGGT-1B', '860abec7937da0a4c03c41d3c269c366e82abdf9')           # (repo, pinned revision)

import os, sys, json, subprocess, hashlib, shutil, base64, gzip, time, zipfile, io
from pathlib import Path
if SMOKE:
    SAVE_TO_DRIVE = False; RUN_NAME += '_smoke'
SCRATCH = Path(os.environ.get('LOOKOUT_SCRATCH', '/content/lookout')); SCRATCH.mkdir(parents=True, exist_ok=True)
REL = Path(os.environ.get('LOOKOUT_RELEASE', SCRATCH / 'release')); MANIFEST = REL / 'release_manifest.json'
DRIVE = None
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/lookout'); DRIVE.mkdir(parents=True, exist_ok=True)
OUT = (DRIVE if DRIVE else SCRATCH) / 'vggt_runs' / RUN_NAME; OUT.mkdir(parents=True, exist_ok=True)

def sh_log(cmd, tail=60):
    """Run; on failure print the LAST lines of the output."""
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        print('\n'.join((r.stdout + r.stderr).splitlines()[-tail:]))
        raise RuntimeError(f'failed ({r.returncode}): {cmd}')
    return r.stdout

## 1 — Development data (6.8 GB; taken from Drive if already there)

In [ ]:
if not MANIFEST.exists():
    tar = (DRIVE / 'lookout-vo-dev-v0.1.tar') if DRIVE else (SCRATCH / 'lookout.tar')
    if not tar.exists():
        sh_log(f'curl -sSL -o {tar} {DATA_URL}')
    sh_log(f'tar -xf {tar} -C {SCRATCH}')
    if not DRIVE:
        tar.unlink()
sh_log(f'cd {REL} && sha256sum --quiet --check CHECKSUMS.sha256')
print({s: sum(c['split'] == s for c in json.load(open(MANIFEST))['clips']) for s in ('train', 'val')})

## 2 — Test data (1.4 GB, SHA-256 checked)

In [ ]:
# ---- held-out TEST package (video, calibration, frame times; no references), as in lookout_speed_v3_1_colab ----
TEST_URL = DATA_URL.replace('lookout-vo-dev-v0.1.tar', 'lookout-vo-test-v0.1.tar')
TREL = Path(os.environ.get('LOOKOUT_TEST_RELEASE', SCRATCH / 'lookout-vo-test-v0.1')); TMAN = TREL / 'release_manifest.json'
if not TMAN.exists():
    tar = (DRIVE / 'lookout-vo-test-v0.1.tar') if DRIVE else (SCRATCH / 'lookout-vo-test-v0.1.tar')
    if not tar.exists():
        sh_log(f'curl -sSL -o {tar} {TEST_URL}')
    want = sh_log(f'curl -sSL {TEST_URL}.sha256').split()[0]; got = sh_log(f'sha256sum {tar}').split()[0]
    assert got == want, f'test archive checksum {got} != published {want}'
    sh_log(f'tar -xf {tar} -C {SCRATCH}')
    if not DRIVE:
        tar.unlink()
    sh_log(f'cd {TREL} && sha256sum --quiet --check CHECKSUMS.sha256')
print(len(json.load(open(TMAN))['clips']), 'test clips')

## 3 — Horizon detector (embedded, sha256; only for MASK_WATER)

In [ ]:
# camera/postprocess.py (horizon detector, used only with MASK_WATER) -- branch claude/lookout-vo, commit 46a4f89.
CODE = SCRATCH / 'code'; (CODE / 'camera').mkdir(parents=True, exist_ok=True)
_PP = ('292a4e81c6f6',
       '...'
)
_data = gzip.decompress(base64.b64decode(''.join(_PP[1:])))
assert hashlib.sha256(_data).hexdigest()[:12] == _PP[0], 'postprocess.py corrupted'
(CODE / 'camera' / 'postprocess.py').write_bytes(_data); print('postprocess.py', len(_data), 'B')


## 4 — VGGT (code and weights pinned)

In [ ]:
# ---- VGGT: code pinned to a commit (no-deps: keep Colab's torch), weights pinned to a revision ----
import torch
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEV == 'cpu':
    print('NO GPU -- Runtime -> Change runtime type -> GPU. (CPU only makes sense for SMOKE.)')
try:
    import vggt
except ImportError:
    sh_log(f'{sys.executable} -m pip install -q --no-deps git+https://github.com/facebookresearch/vggt@{VGGT_COMMIT}')
    sh_log(f'{sys.executable} -m pip install -q einops safetensors huggingface_hub')
from vggt.models.vggt import VGGT
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
DTYPE = (torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16) if DEV == 'cuda' else torch.float32
model = VGGT.from_pretrained(VGGT_WEIGHTS[0], revision=VGGT_WEIGHTS[1]).to(DEV).eval()
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0) if DEV == 'cuda' else 'cpu', '|', DTYPE)

def vggt_poses(batch):
    """batch B x S x 3 x H x W in [0, 1] -> extrinsics B x S x 3 x 4 (OpenCV, camera from world = first frame),
    intrinsics B x S x 3 x 3 -- exactly the README's camera branch."""
    with torch.no_grad():
        with torch.cuda.amp.autocast(dtype=DTYPE, enabled=DEV == 'cuda'):
            tokens, ps_idx = model.aggregator(batch.to(DEV))
        pose_enc = model.camera_head(tokens)[-1]
        ext, intr = pose_encoding_to_extri_intri(pose_enc, batch.shape[-2:])
    return ext.float().cpu().numpy(), intr.float().cpu().numpy()

## 5 — Run (resumes after a disconnect)

In [ ]:
# ---- decode each clip once (starter decoder, ~8 Hz, half resolution = the tool's frames), run every WINDOWS setting ----
import numpy as np, cv2
from PIL import Image
import torchvision.transforms.functional as TF
sys.path.insert(0, str(REL / 'starter'))
from preprocessing import ClipPreprocessor
from public_dataset import load_timestamps
if MASK_WATER:
    sys.path.insert(0, str(CODE / 'camera')); from postprocess import estimate_horizon_line
MAN = {c['key']: (REL, c) for c in json.load(open(MANIFEST))['clips']}
MAN.update({c['key']: (TREL, c) for c in json.load(open(TMAN))['clips']})
def selection():
    if CLIPS:
        return list(CLIPS)
    out = []
    for g in GROUPS:
        sp, cam = g.split(':')
        out += [k for k in sorted(MAN) if MAN[k][1]['split'] == sp and cam in ('*', MAN[k][1]['camera_mode']) and k not in out]
    return out
def prep(bgr):
    """vggt.utils.load_fn.load_and_preprocess_images, mode 'crop', from memory: RGB, width 518, height to a multiple
    of 14 (bicubic), [0, 1]; height centre-cropped to 518 if larger. Optional water mask first."""
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    if MASK_WATER:
        yc, slope, conf = estimate_horizon_line(bgr)
        if conf >= 0.3:                                   # tools/lookout_vo.py HORIZON_MIN_CONF, above_line_mask(+6 px)
            h, w = rgb.shape[:2]; lim = np.maximum(8, yc + slope * (np.arange(w) - w / 2.0) + 6)
            rgb = np.where((np.arange(h)[:, None] < lim[None, :])[..., None], rgb, 0).astype(np.uint8)
    h, w = rgb.shape[:2]; nh = round(h * (518 / w) / 14) * 14
    t = TF.to_tensor(Image.fromarray(rgb).resize((518, nh), Image.Resampling.BICUBIC))
    if nh > 518:
        s = (nh - 518) // 2; t = t[:, s:s + 518]
    return t
def run_clip(k):
    root, c = MAN[k]; t0 = time.time()
    pre = ClipPreprocessor(str(root / c['video']), c['camera_mode'], 8.0, half_res=True, calibration=c['calibration'],
                           frame_timestamps=load_timestamps(str(root), c))
    step = min(kf for kf, _ in WINDOWS.values()); frames, times, dec = {}, [], []
    for i, (_idx, image, _intr, pts, decoded) in enumerate(pre.frames_with_time()):
        times.append(float(pts)); dec.append(int(decoded))
        if i % step == 0:
            frames[i] = prep(image)
    n = len(times); td = time.time() - t0
    for name, (kf, S) in WINDOWS.items():
        span = kf * (S - 1); wins = []
        for s0 in range(0, n - 1, span):
            st = [s0 + j * kf for j in range(S) if s0 + j * kf < n]
            if len(st) >= 3:
                wins.append(st)
        if SMOKE:
            wins = wins[:3]
        E = np.full((len(wins), S, 3, 4), np.nan, np.float32); K = np.full((len(wins), S, 3, 3), np.nan, np.float32)
        steps = np.full((len(wins), S), -1, int)
        for L in sorted(set(map(len, wins))):                       # equal-length windows batch together
            idx = [j for j, w in enumerate(wins) if len(w) == L]
            for b0 in range(0, len(idx), BATCH):
                bi = idx[b0:b0 + BATCH]
                x = torch.stack([torch.stack([frames[s] for s in wins[j]]) for j in bi])
                e, kk = vggt_poses(x)
                for q, j in enumerate(bi):
                    E[j, :L], K[j, :L], steps[j, :L] = e[q], kk[q], wins[j]
        (OUT / name).mkdir(exist_ok=True)
        np.savez_compressed(OUT / name / f'{k}.npz', times=np.asarray(times), dec=np.asarray(dec), steps=steps, ext=E, intr=K,
                            hw=np.asarray(x.shape[-2:]), kf=kf, S=S, masked=MASK_WATER, smoke=SMOKE,
                            vggt=VGGT_COMMIT, weights=VGGT_WEIGHTS[1], camera_mode=c['camera_mode'], split=c['split'])
    print(f'{k} {c["camera_mode"]:8s} {c["split"]:5s} {n} frames, decode {td:4.0f}s, VGGT {time.time() - t0 - td:4.0f}s', flush=True)
sel = selection()
if SMOKE:
    sel = [k for k in sel if MAN[k][1]['camera_mode'] == 'flir-VIS' and MAN[k][1]['split'] == 'train'][:1]
sel = sel[SHARD_INDEX::SHARD_COUNT]
done = lambda k: all((OUT / name / f'{k}.npz').exists() for name in WINDOWS)
todo = [k for k in sel if not done(k)]
print(f'{RUN_NAME}: {len(sel)} clips in this shard, {len(todo)} to run')
log = open(OUT / f'log_shard{SHARD_INDEX}.txt', 'a')
for k in todo:
    try:
        run_clip(k); log.write(f'{k} ok\n')
    except Exception as exc:                                       # one bad clip must not stop the rest
        print(k, 'FAILED', repr(exc)); log.write(f'{k} FAILED {exc!r}\n')
    log.flush()
print(sum(done(k) for k in sel), '/', len(sel), 'clips done')

In [ ]:
# ---- quick look: per clip, VGGT total turn per setting (deg) -- a sign/scale sanity check, not a score ----
import numpy as np
for k in sel:
    row = []
    for name in WINDOWS:
        f = OUT / name / f'{k}.npz'
        if not f.exists():
            continue
        z = np.load(f); tot = 0.0
        for e, st in zip(z['ext'], z['steps']):
            R = [x[:, :3] for x, s in zip(e, st) if s >= 0]
            for a, b in zip(R[:-1], R[1:]):
                Rr = b @ a.T; tot += np.degrees(np.arctan2(Rr[0, 2], Rr[2, 2]))
        row.append(f'{name}: {tot:8.1f}')
    if row:
        print(k, MAN[k][1]['camera_mode'], ' | '.join(row))

## 6 — Package of the results

In [ ]:
# ---- the package to send back: every finished npz (poses only, small) + logs ----
zp = OUT / f'{RUN_NAME}_shard{SHARD_INDEX}.zip'; n = 0
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in WINDOWS:
        for f in sorted((OUT / name).glob('*.npz')):
            z.write(f, f'{name}/{f.name}'); n += 1
    for f in OUT.glob('log_shard*.txt'):
        z.write(f, f'logs/{f.name}')
    z.writestr('README.txt', f'{RUN_NAME} shard {SHARD_INDEX}/{SHARD_COUNT}; WINDOWS {WINDOWS}; MASK_WATER {MASK_WATER}; '
               f'vggt {VGGT_COMMIT}; weights {VGGT_WEIGHTS}\n')
print(f'{zp}: {n} files, {zp.stat().st_size / 1e6:.1f} MB  <- this file goes back')

In [ ]:
# ---- environment lock of this run (next to the results on Drive) ----
import sys, platform, subprocess, pathlib
L = OUT / f'env_lock_{RUN_NAME}.txt'
L.write_text(f"python {sys.version}\nplatform {platform.platform()}\n"
             + subprocess.run(f"{sys.executable} -m pip freeze", shell=True, capture_output=True, text=True).stdout
             + subprocess.run("nvidia-smi --query-gpu=name,driver_version --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)
import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'cudnn', torch.backends.cudnn.version(), file=open(L, 'a'))
print(L, '\n', L.read_text()[:1500])
